# 05 — Genie Agent Deployment

This notebook shows how to deploy the **Data Mesh Genie Space** as a production-ready agent that can be:
1. **Served via Model Serving** — REST API for programmatic access
2. **Integrated with Microsoft Teams** — Chat-based self-service for business users
3. **Orchestrated as a sub-agent** — Part of a multi-agent system via Databricks Apps

**Genie Space:** `Data Mesh Products Self-Service`  
**Space ID:** `01f13923f84d1fb7be77f53f88cb7583`  
**Tables:** `adoit_product.gold.application_landscape`, `snow_product.gold.service_health`, `data_mesh_hub.cross_domain.application_risk_profile`

In [0]:
# Install prerequisites
%pip install "mlflow[databricks]>=3.1.3" databricks-agents>=1.1.0 databricks-sdk>=0.40.0
dbutils.library.restartPython()

## Step 1: Define the Genie ChatAgent

We wrap the Genie Conversation API in an MLflow `ChatAgent`. This creates a standard interface that:
- Accepts natural language questions about our data products
- Routes them to the Genie space (which has access to all 3 gold tables)
- Returns structured responses with query results or descriptions

The agent handles the full conversation lifecycle: start → poll → extract response.

In [0]:
# Write the Genie ChatAgent model to a file
agent_code = '''
import time, uuid, mlflow
from mlflow.pyfunc import ChatAgent
from mlflow.types.agent import ChatAgentMessage, ChatAgentResponse, ChatContext
from databricks.sdk import WorkspaceClient
from typing import Any

GENIE_SPACE_ID = "01f13923f84d1fb7be77f53f88cb7583"

class DataMeshGenieAgent(ChatAgent):
    """ChatAgent that wraps the Data Mesh Genie Space for self-service analytics.
    
    Supports questions across all three data products:
    - Application Landscape (EA domain)
    - Service Health (ITSM domain)
    - Application Risk Profile (cross-domain)
    """

    def __init__(self):
        self.workspace_client = WorkspaceClient()

    def predict(
        self,
        messages: list[ChatAgentMessage],
        context: ChatContext | None = None,
        custom_inputs: dict[str, Any] | None = None,
    ) -> ChatAgentResponse:
        last_message = messages[-1].content if messages else ""

        genie = self.workspace_client.genie
        conv = genie.start_conversation(space_id=GENIE_SPACE_ID, content=last_message)

        # Poll until Genie completes (up to 2 min)
        for _ in range(60):
            msg = genie.get_message(
                space_id=GENIE_SPACE_ID,
                conversation_id=conv.conversation_id,
                message_id=conv.message_id,
            )
            if msg.status in ("COMPLETED", "FAILED"):
                break
            time.sleep(2)

        # Extract response
        response_text = "I could not get a response from the data products."
        if msg.status == "COMPLETED" and msg.attachments:
            for att in msg.attachments:
                if att.text:
                    response_text = att.text.content
                    break
                if att.query:
                    response_text = f"Query: {att.query.query}\\n\\nDescription: {att.query.description or ''}"
                    break

        return ChatAgentResponse(
            messages=[ChatAgentMessage(id=str(uuid.uuid4()), role="assistant", content=response_text)]
        )

mlflow.models.set_model(DataMeshGenieAgent())
'''

agent_path = '/Workspace/Users/skarotirajashekar@godevsuite060.onmicrosoft.com/data_mesh_demo/genie_agent_model.py'
with open(agent_path, 'w') as f:
    f.write(agent_code)
print(f'Agent code written to: {agent_path}')

## Step 2: Log & Register in Unity Catalog

We log the agent as an MLflow model and register it in Unity Catalog. This gives us:
- **Versioning** — Track agent iterations
- **Lineage** — UC knows the agent depends on the Genie space
- **Governance** — Same access controls as any UC object

In [0]:
import mlflow
from mlflow.models.resources import DatabricksGenieSpace

mlflow.set_registry_uri('databricks-uc')

UC_MODEL_NAME = 'data_mesh_hub.cross_domain.data_mesh_genie_agent'

input_example = {
    'messages': [{'role': 'user', 'content': 'Which applications have the highest composite risk score?'}]
}

resources = [DatabricksGenieSpace(genie_space_id='01f13923f84d1fb7be77f53f88cb7583')]

with mlflow.start_run(run_name='data_mesh_genie_agent'):
    logged = mlflow.pyfunc.log_model(
        name='genie-agent',
        python_model='/Workspace/Users/skarotirajashekar@godevsuite060.onmicrosoft.com/data_mesh_demo/genie_agent_model.py',
        input_example=input_example,
        resources=resources,
        pip_requirements=['mlflow>=3.1.3', 'databricks-sdk>=0.40.0'],
    )
    print(f'MLflow Run: {logged.run_id}')
    print(f'Model URI: {logged.model_uri}')

uc_model = mlflow.register_model(model_uri=logged.model_uri, name=UC_MODEL_NAME)
print(f'Registered: {UC_MODEL_NAME} v{uc_model.version}')

## Step 3: Deploy to Model Serving

Deploy the registered agent to a Databricks Model Serving endpoint. Once live, any application can send natural language questions via REST API and get answers grounded in the data mesh gold products.

In [0]:
from databricks import agents

deployment = agents.deploy(
    model_name=UC_MODEL_NAME,
    model_version=uc_model.version,
)

print(f'Endpoint: {deployment.endpoint_name}')
print(f'URL: {deployment.endpoint_url}')
print(f'\n=== Save this endpoint name for Teams integration ===')

## Step 4: Multi-Agent Orchestrator (Databricks Apps)

For a richer experience, deploy an **OpenAI Agents SDK multi-agent orchestrator** as a Databricks App. This allows combining the Genie data agent with other specialist agents (RAG, knowledge base, etc.).

### Configuration

Clone the [Databricks app-templates](https://github.com/databricks/app-templates) repo and edit `agent_server/agent.py`:

In [0]:
# Multi-agent orchestrator configuration for Databricks Apps
# Edit agent_server/agent.py in the cloned template with this config:

SUBAGENTS = [
    {
        'name': 'data_mesh_analyst',
        'type': 'genie',
        'space_id': '01f13923f84d1fb7be77f53f88cb7583',
        'description': (
            'Query the Data Mesh for application landscape, service health, '
            'and risk profile data. Use for questions about enterprise applications, '
            'incidents, changes, SLA compliance, and composite risk scores.'
        ),
    },
    # Add more sub-agents as needed:
    # {
    #     'name': 'policy_advisor',
    #     'type': 'serving_endpoint',
    #     'endpoint': 'rag-policy-docs',
    #     'description': 'Answer questions about IT governance policies and standards.',
    # },
]

print(f'Configured {len(SUBAGENTS)} subagent(s): {[s["name"] for s in SUBAGENTS]}')
print()
print('Deploy with Databricks Asset Bundles:')
print('  1. cd app-templates/agent-openai-agents-sdk-multiagent')
print('  2. databricks bundle validate')
print('  3. databricks bundle deploy')
print('  4. databricks bundle run agent_openai_agents_sdk_multiagent')

## Step 5: Microsoft Teams Integration

Connect the serving endpoint to Microsoft Teams so business users can chat with data products directly:

1. **Save the endpoint name** from Step 3
2. **Create Azure Bot Service** — Azure Portal → Bot Service + App Service
3. **Create OAuth federation policy** — `databricks account federation-policy create` via CLI
4. **Deploy bot code** — Clone [databricks-bot-service](https://github.com/databricks/databricks-bot-service) → Azure App Service
5. **Test in Teams** — Verify auth and ask: *"Which applications have the highest risk?"*

**Full guide:** [Connect an AI agent to Microsoft Teams](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-framework/teams-agent/)

---

### Architecture Summary

```
Teams User → Azure Bot Service → Model Serving Endpoint → ChatAgent → Genie Space → Unity Catalog
                                                                         ↓
                                                              ┌──────────────────────┐
                                                              │  Gold Data Products   │
                                                              │  • app_landscape      │
                                                              │  • service_health     │
                                                              │  • risk_profile       │
                                                              └──────────────────────┘
```